# DS006104 (2021) — stepwise EEG and trial-alignment analysis

This notebook analyses the locally downloaded `S01–S16 / ses-02` EEG subset without altering the source EDF files. It is designed for the later EEG-to-audio reconstruction pipeline.

## Execution plan

1. Audit the BIDS tree and flag incomplete S3-transfer remnants.
2. Build a recording and stimulus-trial index from BIDS filenames and `events.tsv`.
3. Load one representative recording lazily, inspect channels and timing, then create basic quality-control figures.
4. Create a stimulus-locked trial window figure: a coloured analysis timeline above offset EEG traces, matching the style of the KaraOne/FEIS examples.
5. Produce task/category summaries, inter-stimulus timing, PSD, and an event-locked average.
6. Audit the optional internal-audio mapping needed for future EEG-to-audio training.

Run cells in order. The notebook only loads a small channel/time subset for plotting; it does **not** run ICA, reject channels, or overwrite raw data.

In [ ]:
from pathlib import Path
import os
import re
import warnings

# Avoid a known local numba cache issue; this notebook only processes short QC segments.
os.environ.setdefault('NUMBA_DISABLE_JIT', '1')

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal

mne.set_log_level('WARNING')
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.spines.top': False, 'axes.spines.right': False})

def locate_bundle_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'ds006104').exists():
            return candidate
    fallback = Path('/Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/eeg2wave_server_bundle/eeg-recon-0809')
    if (fallback / 'data' / 'ds006104').exists():
        return fallback
    raise FileNotFoundError('Could not find eeg-recon-0809/data/ds006104. Set BUNDLE_ROOT manually.')

BUNDLE_ROOT = locate_bundle_root()
DATA_ROOT = BUNDLE_ROOT / 'data' / 'ds006104'
FIG_DIR = BUNDLE_ROOT / 'reports' / 'generated' / 'ds006104_2021'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Bundle root: {BUNDLE_ROOT}')
print(f'Data root:   {DATA_ROOT}')
print(f'Figures:     {FIG_DIR}')

## 1. Inventory and transfer audit

A prior interrupted `aws s3 sync` can leave a temporary file such as `*.edf.<random>`. This cell reports it but never removes it. The formal `*_eeg.edf` files remain the analysis inputs.

In [ ]:
temp_transfer_files = sorted(DATA_ROOT.rglob('*.edf.*'))
if temp_transfer_files:
    print('WARNING: possible incomplete transfer remnants (do not analyse these):')
    for path in temp_transfer_files[:20]:
        print(' -', path.relative_to(DATA_ROOT), f'({path.stat().st_size / 2**20:.1f} MiB)')
else:
    print('No EDF transfer remnants detected.')

required_top_level = ['dataset_description.json', 'participants.tsv', 'participants.json']
for filename in required_top_level:
    print(f"{'OK' if (DATA_ROOT / filename).exists() else 'MISSING'}  {filename}")

## 2. Build a recording index

The index collects BIDS paths, task identity, EDF size, and number of `stimulus` rows. It is intentionally generated from the local tree instead of hard-coded.

In [ ]:
def task_from_name(name):
    match = re.search(r'_task-([^_]+)_', name)
    return match.group(1) if match else 'unknown'

rows = []
for eeg_json in sorted(DATA_ROOT.glob('sub-S*/ses-02/eeg/*_eeg.json')):
    eeg_dir = eeg_json.parent
    subject = eeg_json.parts[-4].replace('sub-', '')
    task = task_from_name(eeg_json.name)
    stem = eeg_json.name.replace('_eeg.json', '')
    edf = eeg_dir / f'{stem}_eeg.edf'
    events = eeg_dir / f'{stem}_events.tsv'
    channels = eeg_dir / f'{stem}_channels.tsv'
    event_df = pd.read_csv(events, sep='\t') if events.exists() else pd.DataFrame()
    n_stimulus = int((event_df.get('trial_type', pd.Series(dtype=str)) == 'stimulus').sum())
    rows.append({
        'subject': subject, 'session': '02', 'task': task, 'edf': edf, 'events': events, 'channels': channels,
        'edf_gib': edf.stat().st_size / 2**30 if edf.exists() else np.nan,
        'n_event_rows': len(event_df), 'n_stimulus_rows': n_stimulus,
    })

recordings = pd.DataFrame(rows).sort_values(['subject', 'task']).reset_index(drop=True)
assert not recordings.empty, 'No 2021 EEG recordings found under data/ds006104.'
display(recordings[['subject', 'task', 'edf_gib', 'n_event_rows', 'n_stimulus_rows']])
print(f"Recordings: {len(recordings)} | Subjects: {recordings.subject.nunique()} | EDF volume: {recordings.edf_gib.sum():.1f} GiB")

## 3. Build a stimulus-trial index

A `stimulus` event is the trial anchor. The nearest preceding `TMS` event is retained when it is within 250 ms. This creates a transparent alignment table for later matching to the private audio manifest.

In [ ]:
def clean_token(value):
    if pd.isna(value):
        return ''
    value = str(value).replace('\x00', '').strip()
    return '' if value.lower() in {'n/a', 'nan'} else value

def make_trial_index(recording_table):
    trial_tables = []
    for record in recording_table.itertuples(index=False):
        events = pd.read_csv(record.events, sep='\t')
        stimulus = events.loc[events['trial_type'].eq('stimulus')].copy().reset_index(names='event_row')
        tms_rows = events.loc[events['trial_type'].eq('TMS')].copy().reset_index(names='tms_event_row')
        paired_tms = []
        for onset in stimulus['onset'].to_numpy(float):
            eligible = tms_rows.loc[tms_rows['onset'].le(onset)]
            candidate = eligible.iloc[-1] if len(eligible) else None
            paired_tms.append(candidate if candidate is not None and onset - float(candidate['onset']) <= 0.25 else None)
        stimulus['tms_onset'] = [np.nan if row is None else row['onset'] for row in paired_tms]
        for column in ['trial', 'category', 'tms_target', 'tms_intensity']:
            paired_value = [np.nan if row is None or column not in row.index else row[column] for row in paired_tms]
            if column in stimulus:
                stimulus[column] = pd.Series(paired_value).combine_first(stimulus[column].reset_index(drop=True))
            else:
                stimulus[column] = paired_value
        phoneme_cols = [col for col in ['phoneme1', 'phoneme2', 'phoneme3'] if col in stimulus]
        stimulus['phoneme_label'] = stimulus[phoneme_cols].apply(lambda r: ''.join(clean_token(x) for x in r), axis=1)
        stimulus['tms_to_stim_s'] = stimulus['onset'] - stimulus['tms_onset']
        stimulus['subject'] = record.subject
        stimulus['session'] = record.session
        stimulus['task'] = record.task
        stimulus['edf'] = str(record.edf)
        stimulus['trial_in_recording'] = np.arange(len(stimulus))
        trial_tables.append(stimulus)
    return pd.concat(trial_tables, ignore_index=True)

trial_index = make_trial_index(recordings)
trial_index.to_csv(FIG_DIR / 'stimulus_trial_index.csv', index=False)
display(trial_index[['subject', 'task', 'trial_in_recording', 'onset', 'category', 'phoneme_label', 'tms_to_stim_s']].head(12))
print(f'Stimulus trials indexed: {len(trial_index):,}')

## 4. Dataset-level visualisation

The left panel checks stimulus-trial balance by task and lexical category. The right panel checks the distribution of inter-stimulus intervals, which is useful for choosing a leakage-free reconstruction window.

In [ ]:
ordered_tasks = sorted(trial_index['task'].unique())
category_counts = pd.crosstab(trial_index['task'], trial_index['category']).reindex(ordered_tasks).fillna(0)
isi_table = trial_index.sort_values(['subject', 'task', 'onset']).copy()
isi_table['isi_s'] = isi_table.groupby(['subject', 'task'])['onset'].diff()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
category_counts.plot.bar(ax=axes[0], color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
axes[0].set_xlabel('Task')
axes[0].set_ylabel('Stimulus trials')
axes[0].legend(title='Category', frameon=False)

for task, group in isi_table.dropna(subset=['isi_s']).groupby('task'):
    axes[1].hist(group['isi_s'], bins=35, alpha=0.55, label=task)
axes[1].set_xlabel('Inter-stimulus interval (s)')
axes[1].set_ylabel('Count')
axes[1].legend(frameon=False)

fig.savefig(FIG_DIR / 'dataset_trial_inventory.png', bbox_inches='tight')
plt.show()

## 5. Select a representative recording and inspect the raw EDF

`Words` is selected by default because it contains phoneme triplets. Change `EXAMPLE_SUBJECT` or `EXAMPLE_TASK` if another recording is preferable. EDF is opened with `preload=False`, so the full 35 GB dataset is never loaded into memory.

In [ ]:
EXAMPLE_SUBJECT = 'S01'
EXAMPLE_TASK = 'Words'
EXAMPLE_TRIAL = 0  # index among stimulus rows in this recording

example_record = recordings.query('subject == @EXAMPLE_SUBJECT and task == @EXAMPLE_TASK').iloc[0]
raw = mne.io.read_raw_edf(example_record.edf, preload=False, infer_types=True, verbose=False)
example_trials = trial_index.query('subject == @EXAMPLE_SUBJECT and task == @EXAMPLE_TASK').reset_index(drop=True)
example_trial = example_trials.iloc[EXAMPLE_TRIAL]

print(f"EDF: {example_record.edf.name}")
print(f"Sampling rate: {raw.info['sfreq']:.0f} Hz | Channels: {len(raw.ch_names)} | Duration: {raw.times[-1] / 60:.1f} min")
print('Example trial:')
display(example_trial[['trial_in_recording', 'onset', 'category', 'phoneme_label', 'tms_onset', 'tms_to_stim_s', 'tms_target']].to_frame().T)
print('First channels:', raw.ch_names[:12])

## 6. Trial-window figure for visual QC

The figure follows the requested KaraOne/FEIS design: a coloured timeline above, then stacked EEG traces below. The coloured ranges are **analysis conventions**, not a claim about exact audio duration: baseline (−0.5–0 s), early stimulus-locked (0–0.5 s), acoustic/phonemic candidate window (0.5–1.5 s), and late/response window (1.5–2.5 s). The actual TMS marker is shown when available.

In [ ]:
def available_channels(raw_obj, preferred=('Fp1', 'Fz', 'Cz', 'Pz', 'O1', 'T7')):
    lookup = {name.lower(): name for name in raw_obj.ch_names}
    return [lookup[name.lower()] for name in preferred if name.lower() in lookup]

def plot_trial_window(raw_obj, trial, output_path, channels=None, pre_s=0.5, post_s=2.5):
    channels = channels or available_channels(raw_obj)
    if len(channels) < 3:
        channels = raw_obj.ch_names[:min(6, len(raw_obj.ch_names))]
    onset = float(trial['onset'])
    crop_start = max(0.0, onset - pre_s)
    crop_stop = min(float(raw_obj.times[-1]), onset + post_s)
    view = raw_obj.copy().pick(channels).crop(tmin=crop_start, tmax=crop_stop).load_data()
    view.filter(l_freq=1.0, h_freq=30.0, method='iir', verbose=False)
    values_uv = view.get_data() * 1e6
    rel_time = view.times + crop_start - onset
    baseline = values_uv[:, rel_time < 0]
    if baseline.size:
        values_uv = values_uv - np.median(baseline, axis=1, keepdims=True)
    spread = np.nanmedian(np.ptp(values_uv, axis=1))
    offset = max(25.0, spread * 1.25)

    fig = plt.figure(figsize=(15, 8.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 1, height_ratios=[1, 3])
    ax_timeline = fig.add_subplot(grid[0])
    ax_eeg = fig.add_subplot(grid[1], sharex=ax_timeline)
    phases = [(-pre_s, 0.0, '#7D96AC', 'baseline'), (0.0, 0.5, '#F6C85F', 'early\nstimulus'),
              (0.5, 1.5, '#9BB8AC', 'acoustic /\nphonemic'), (1.5, post_s, '#EF8A8A', 'late /\nresponse')]
    for start, stop, color, label in phases:
        ax_timeline.axvspan(start, stop, color=color, alpha=0.95)
        ax_eeg.axvspan(start, stop, color=color, alpha=0.13)
        ax_timeline.text((start + stop) / 2, 0.5, label, ha='center', va='center', fontsize=12)
    ax_timeline.set_ylim(0, 1)
    ax_timeline.set_yticks([])
    ax_timeline.tick_params(labelbottom=False)

    for idx, (channel, series) in enumerate(zip(channels, values_uv)):
        ax_eeg.plot(rel_time, series + idx * offset, lw=0.85, label=channel)
    ax_eeg.axvline(0, color='black', lw=1.1, ls='--', label='stimulus onset')
    if pd.notna(trial.get('tms_onset', np.nan)):
        tms_rel = float(trial['tms_onset']) - onset
        if -pre_s <= tms_rel <= post_s:
            ax_eeg.axvline(tms_rel, color='#6A3D9A', lw=1.2, ls=':', label='TMS')
    annotation = (f"subject={trial['subject']}  task={trial['task']}  trial={int(trial['trial_in_recording'])}\n"
                  f"category={trial.get('category', 'n/a')}  phonemes={trial.get('phoneme_label', '') or 'n/a'}\n"
                  'Coloured bands are analysis windows; they are not measured audio boundaries.')
    ax_eeg.text(0.01, 0.98, annotation, transform=ax_eeg.transAxes, va='top', ha='left',
                bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.9})
    ax_eeg.set_xlabel('Time relative to stimulus onset (s)')
    ax_eeg.set_ylabel('EEG amplitude (µV, offset)')
    ax_eeg.legend(loc='upper right', ncol=2, frameon=True)
    fig.suptitle('DS006104 stimulus-locked example trial', fontsize=15)
    fig.savefig(output_path, bbox_inches='tight')
    return fig

trial_figure_path = FIG_DIR / f"trial_window_{EXAMPLE_SUBJECT}_{EXAMPLE_TASK}_trial-{EXAMPLE_TRIAL:03d}.png"
fig = plot_trial_window(raw, example_trial, trial_figure_path)
print(f'Saved: {trial_figure_path}')
plt.show()

## 7. PSD and stimulus-locked average

This is deliberately a lightweight QC analysis: six channels and at most 100 stimulus trials from one recording. It should reveal line-noise dominance, amplitude anomalies, and gross stimulus-locked structure before any full preprocessing. It is not a substitute for artifact rejection/ICA.

In [ ]:
PLOT_CHANNELS = available_channels(raw)
MAX_EPOCHS = 100
TMIN, TMAX = -0.2, 0.8

def extract_epochs(raw_obj, onsets, channels, tmin, tmax, max_epochs=100):
    sfreq = float(raw_obj.info['sfreq'])
    n_before, n_after = int(round(-tmin * sfreq)), int(round(tmax * sfreq))
    epochs = []
    for onset in np.asarray(onsets)[:max_epochs]:
        center = int(raw_obj.time_as_index(onset)[0])
        start, stop = center - n_before, center + n_after
        if start < 0 or stop > raw_obj.n_times:
            continue
        values = raw_obj.get_data(picks=channels, start=start, stop=stop) * 1e6
        values = values - values[:, :n_before].mean(axis=1, keepdims=True)
        epochs.append(values)
    return np.stack(epochs), np.arange(-n_before, n_after) / sfreq

epochs_uv, epoch_times = extract_epochs(raw, example_trials['onset'], PLOT_CHANNELS, TMIN, TMAX, MAX_EPOCHS)
evoked_uv = epochs_uv.mean(axis=0)
sem_uv = epochs_uv.std(axis=0, ddof=1) / np.sqrt(len(epochs_uv))

qc_segment = raw.copy().pick(PLOT_CHANNELS).crop(tmin=float(example_trial['onset']), tmax=float(example_trial['onset']) + 20).load_data()
freqs, psd = signal.welch(qc_segment.get_data() * 1e6, fs=raw.info['sfreq'], nperseg=min(4096, qc_segment.n_times), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for channel, trace, err in zip(PLOT_CHANNELS, evoked_uv, sem_uv):
    axes[0].plot(epoch_times, trace, lw=1.3, label=channel)
axes[0].axvline(0, color='black', ls='--', lw=1)
axes[0].set(xlabel='Time from stimulus onset (s)', ylabel='Baseline-corrected amplitude (µV)', title=f'Stimulus-locked average (n={len(epochs_uv)})')
axes[0].legend(frameon=False, ncol=2)
for channel, spectrum in zip(PLOT_CHANNELS, psd):
    axes[1].plot(freqs, 10 * np.log10(spectrum + np.finfo(float).eps), lw=1.1, label=channel)
axes[1].set(xlim=(1, 80), xlabel='Frequency (Hz)', ylabel='Power (dB µV²/Hz)', title='20-second raw-EEG PSD')
axes[1].axvline(60, color='gray', ls=':', lw=1)
axes[1].legend(frameon=False, ncol=2)
fig.savefig(FIG_DIR / 'stimulus_locked_qc.png', bbox_inches='tight')
plt.show()

## 8. Internal-audio audit for EEG-to-audio reconstruction

The public DS006104 download supplies EEG/events, while the acoustic targets are your internal files. Place a mapping file at `data/ds006104/audio_internal/trial_audio_manifest.csv` with columns `subject, session, task, trial, audio_path`. The notebook pairs each stimulus to the preceding TMS row (50 ms earlier in this recording) and uses that paired row's `trial` value for the audio join.

In [ ]:
AUDIO_DIR = DATA_ROOT / 'audio_internal'
AUDIO_MANIFEST = AUDIO_DIR / 'trial_audio_manifest.csv'
required_audio_columns = {'subject', 'session', 'task', 'trial', 'audio_path'}

if AUDIO_MANIFEST.exists():
    audio_manifest = pd.read_csv(AUDIO_MANIFEST)
    missing_columns = required_audio_columns.difference(audio_manifest.columns)
    if missing_columns:
        raise ValueError(f'Audio manifest is missing columns: {sorted(missing_columns)}')
    display(audio_manifest.head())
    print(f'Audio manifest rows: {len(audio_manifest):,}')
else:
    print('No internal-audio manifest found yet. Expected path:')
    print(AUDIO_MANIFEST)
    print('Required columns:', ', '.join(sorted(required_audio_columns)))
    print('Example: S01,02,Words,351,audio/S01_Words_trial351.wav')

## Next analysis step

Before training a reconstruction model, inspect the saved figures and verify (i) EDF file integrity, (ii) whether stimulus/TMS timing is consistent, and (iii) that internal audio can be joined one-to-one to the BIDS `trial` identifier. Full filtering, bad-channel handling, and epoch rejection should only be run after this QC review.